# Week 2, Day 4 Bonus: Python Foundations Drills (Intermediate)

Welcome to a slow, confidence-building rep session. Day 2 moved fast. This notebook
slows down and gives you more practice on the Python that data engineers use every
day, plus a tour of the standard library modules you will reach for constantly:
`math`, `statistics`, `random`, `datetime` / `timedelta`, `calendar`, `pathlib`,
`glob`, and `itertools`.

**How this notebook works.** Each drill has four parts:

1. **Concept.** A short explanation of the technique or module and a link to the
   official docs. Bookmark these. Reading real docs is a core skill.
2. **Example.** One or two small runnable cells you can read and run to see the idea
   in action.
3. **Your task.** A clearly stated exercise.
4. **Starter cell.** Code with `# TODO` markers for you to complete. Solutions live
   in the `solutions/` folder, but do not open them until you have tried.

**Ground rules for this notebook:**

- **AI-Free Zone (Weeks 1 to 4).** Do not use Copilot, ChatGPT, or any AI assistant
  to write this code. The point is to build the muscle yourself. Use the linked docs
  and the hints.
- Work top to bottom. The setup cell below defines shared sample data used by later
  drills, so run it first.
- Run every cell. Read the output. If it errors, read the traceback before changing
  anything.

## Setup

Copy this whole `Python Drills` folder into your `student-work/week2/` project (do
not edit the provided copy in place), then open this notebook there and select your
`student-work/week2/.venv` as the kernel. This notebook uses only the Python
standard library, so there is nothing to `uv add`. If you have not created the
day project yet:

```bash
cd student-work/week2
uv init          # only if you have not already
# no extra packages needed for these drills
```

Run the next cell first. It creates a small `sample_data/` folder of claim files
next to this notebook and defines a shared `CLAIMS` list that several drills reuse.


In [ ]:
# SETUP: run this first. It creates sample files and shared data used across drills.
import csv
from pathlib import Path

# A shared, in-memory dataset of insurance claims used by several drills below.
CLAIMS = [
    {"claim_id": "CLM-1001", "state": "CT", "line": "auto", "amount": 4200.0,  "status": "open",   "opened": "2026-05-02", "closed": "2026-05-20"},
    {"claim_id": "CLM-1002", "state": "CT", "line": "home", "amount": 15800.0, "status": "open",   "opened": "2026-05-04", "closed": "2026-05-26"},
    {"claim_id": "CLM-1003", "state": "MA", "line": "auto", "amount": 2600.0,  "status": "closed", "opened": "2026-05-05", "closed": "2026-05-08"},
    {"claim_id": "CLM-1004", "state": "NY", "line": "auto", "amount": 9100.0,  "status": "open",   "opened": "2026-06-01", "closed": "2026-06-15"},
    {"claim_id": "CLM-1005", "state": "MA", "line": "home", "amount": 22300.0, "status": "closed", "opened": "2026-06-03", "closed": "2026-06-30"},
    {"claim_id": "CLM-1006", "state": "NY", "line": "home", "amount": 530.0,   "status": "open",   "opened": "2026-06-09", "closed": "2026-06-12"},
    {"claim_id": "CLM-1007", "state": "CT", "line": "auto", "amount": 7400.0,  "status": "closed", "opened": "2026-07-01", "closed": "2026-07-05"},
]

# Write the claims to monthly CSV files so the glob/pathlib drills have real files.
DATA_DIR = Path("sample_data")
(DATA_DIR / "archive").mkdir(parents=True, exist_ok=True)

fields = ["claim_id", "state", "line", "amount", "status", "opened", "closed"]

def _write_month(path, rows):
    with open(path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)

_write_month(DATA_DIR / "claims_2026-05.csv", [c for c in CLAIMS if c["opened"].startswith("2026-05")])
_write_month(DATA_DIR / "claims_2026-06.csv", [c for c in CLAIMS if c["opened"].startswith("2026-06")])
_write_month(DATA_DIR / "claims_2026-07.csv", [c for c in CLAIMS if c["opened"].startswith("2026-07")])
# An older month lives in an archive subfolder (used by the recursive glob drill).
_write_month(DATA_DIR / "archive" / "claims_2026-04.csv", [
    {"claim_id": "CLM-0999", "state": "CT", "line": "auto", "amount": 3300.0, "status": "closed", "opened": "2026-04-10", "closed": "2026-04-19"},
])
(DATA_DIR / "readme.txt").write_text("Sample claim extracts for the Python Drills.\n")

print("Setup complete. Files created under:", DATA_DIR.resolve())
for p in sorted(DATA_DIR.rglob("*")):
    print(" -", p)
print("\nCLAIMS in memory:", len(CLAIMS), "records")

---
# Part 1: Foundations

Before the modules, a few reps on the building blocks: functions, loops, and nested
data structures. These are the three things you will use in every single drill and
every pipeline you ever write.


## Drill 1: Functions

**Concept.** A function packages logic behind a name so you can reuse it and test it.
You define parameters (inputs), optionally give them defaults, do work, and `return`
a result. A short `"""docstring"""` on the first line documents what it does.

Docs: https://docs.python.org/3/tutorial/controlflow.html#defining-functions

Why a data engineer cares: pipelines are just many small, well-named functions
(clean a column, parse a date, compute a metric) composed together.


In [ ]:
# Example: a function with a docstring, a typed idea, and a return value.
def loss_ratio(paid, earned_premium):
    """Return paid losses divided by earned premium, as a fraction."""
    return paid / earned_premium

print(loss_ratio(70, 100))          # 0.7
print(round(loss_ratio(70, 100), 2))

In [ ]:
# Example: a default parameter value. Callers can omit `ndigits`.
def as_percent(fraction, ndigits=1):
    """Format a fraction like 0.732 as a percent string like '73.2%'."""
    return f"{round(fraction * 100, ndigits)}%"

print(as_percent(0.732))       # 73.2%
print(as_percent(0.732, 0))    # 73%

**Your task.**

1. Write `flag_large_claim(amount, threshold=10000)` that returns `True` when the
   amount is at or above the threshold, otherwise `False`. Give `threshold` a default
   so a caller can leave it out. This practices one thing: a default parameter and
   returning a boolean.
2. Write `summarize(amounts)` that takes a list of numbers and returns a dict with
   keys `count`, `total`, and `average` (average rounded to 2 decimals). Return
   zeros for an empty list instead of crashing.

### Expected output

```text
True
False
True
{'count': 3, 'total': 600, 'average': 200.0}
{'count': 0, 'total': 0, 'average': 0}
```


In [ ]:
def flag_large_claim(amount, threshold=10000):
    """Return True if amount is at or above threshold (default 10000)."""
    # TODO: return the comparison result. This is a one-liner.
    pass

def summarize(amounts):
    """Return {'count', 'total', 'average'} for a list of numbers. Empty -> zeros."""
    # TODO: handle the empty list first (return count/total/average of 0).
    # TODO: otherwise compute count, total, and the rounded average.
    pass

# Quick checks (uncomment once implemented):
# print(flag_large_claim(15800))                 # True
# print(flag_large_claim(530))                   # False
# print(flag_large_claim(9000, threshold=5000))  # True
# print(summarize([100, 200, 300]))              # {'count': 3, 'total': 600, 'average': 200.0}
# print(summarize([]))                           # {'count': 0, 'total': 0, 'average': 0}

## Drill 2: For-loops

**Concept.** A `for` loop walks through the items of any iterable (list, dict,
string, file). Three loop helpers show up constantly:

- `range(n)` gives you the numbers `0..n-1`.
- `enumerate(items)` gives you `(index, item)` pairs.
- `zip(a, b)` walks two sequences in lockstep.

Docs: https://docs.python.org/3/tutorial/controlflow.html#for-statements and
https://docs.python.org/3/library/functions.html#enumerate

Why a data engineer cares: iterating rows and accumulating a running total or a
per-key tally is the heart of "by hand" aggregation, which you will later hand off
to pandas or SQL.


In [ ]:
# Example: enumerate gives you a counter for free (start at 1 here).
lines = ["auto", "home", "auto"]
for i, line in enumerate(lines, start=1):
    print(i, line)

In [ ]:
# Example: zip walks two lists together, and a running total accumulates in a loop.
ids = ["CLM-1", "CLM-2", "CLM-3"]
amounts = [100, 250, 75]

running = 0
for claim_id, amount in zip(ids, amounts):
    running += amount
    print(f"{claim_id}: +{amount} -> running total {running}")

**Your task.** Using the shared `CLAIMS` list from setup and a `for` loop (no
pandas, no comprehensions here, the point is loop mechanics):

1. Build `total_by_state`, a dict mapping each state to the sum of its claim
   `amount` values.
2. Count how many claims have `status == "open"` into `open_count`.
3. Print a numbered roster of claim ids using `enumerate(..., start=1)`.

### Expected output

```text
total_by_state: {'CT': 27400.0, 'MA': 24900.0, 'NY': 9630.0}
open_count: 4
roster:
  1. CLM-1001
  2. CLM-1002
  3. CLM-1003
  4. CLM-1004
  5. CLM-1005
  6. CLM-1006
  7. CLM-1007
```


In [ ]:
total_by_state = {}
open_count = 0

for claim in CLAIMS:
    # TODO: add this claim's amount to total_by_state[claim["state"]].
    #       Remember the key may not exist yet. dict.get(key, 0) helps.
    # TODO: if the status is "open", increment open_count.
    pass

print("total_by_state:", total_by_state)
print("open_count:", open_count)

print("roster:")
# TODO: loop with enumerate(CLAIMS, start=1) and print "  {n}. {claim_id}".

## Drill 3: Nested data structures

**Concept.** Real records are nested: a list of dicts, a dict whose values are lists,
or dicts inside dicts. You navigate them by chaining `[]` access and using
`.get(key, default)` to avoid `KeyError` on missing keys.

Docs: https://docs.python.org/3/tutorial/datastructures.html

Why a data engineer cares: JSON from an API (Day 3) is exactly this shape. Being
comfortable reaching into nested structures is non-negotiable.


In [ ]:
# Example: a dict of dicts, and safe access with .get.
claim = {
    "claim_id": "CLM-2001",
    "amount": 5000,
    "adjuster": {"name": "Rivera", "office": "Hartford"},
}
print(claim["adjuster"]["name"])          # Rivera
print(claim.get("line", "unknown"))       # unknown  (key missing -> default)

In [ ]:
# Example: grouping into a dict of lists (bucket claim ids under their state).
by_state = {}
for c in CLAIMS:
    by_state.setdefault(c["state"], []).append(c["claim_id"])
print(by_state)

**Your task.** Build a nested summary. From `CLAIMS`, produce `summary`, a dict that
maps each state to an inner dict with two keys: `count` (how many claims) and
`total` (sum of amounts). Then read one value back out of the nested structure.

### Expected output

```text
{'CT': {'count': 3, 'total': 27400.0}, 'MA': {'count': 2, 'total': 24900.0}, 'NY': {'count': 2, 'total': 9630.0}}
MA total: 24900.0
```


In [ ]:
summary = {}

for c in CLAIMS:
    state = c["state"]
    # TODO: if state not seen yet, initialize summary[state] = {"count": 0, "total": 0}.
    # TODO: increment that state's count by 1 and add the amount to its total.
    pass

print(summary)

# TODO: print the total dollars for "MA" by reaching into the nested dict.
# print("MA total:", ...)

---
# Part 2: Standard library modules

Python ships with a "batteries included" standard library. Below are the modules a
data engineer reaches for before installing anything. For each one: read the concept,
run the example, then complete the task.


## Drill 4: `math`

**Concept.** `math` gives you numeric helpers that are more correct or more precise
than writing your own: `math.ceil`, `math.floor`, `math.sqrt`, `math.isclose`
(compare floats safely), and `math.fsum` (accurate float summation).

Docs: https://docs.python.org/3/library/math.html

Why a data engineer cares: never compare floats with `==`. Use `math.isclose`.
Rounding decisions (ceil vs floor) matter for billing, batching, and pagination.


In [ ]:
import math

print(math.ceil(4.1))              # 5
print(math.floor(4.9))             # 4
print(round(math.sqrt(2), 4))      # 1.4142
print(0.1 + 0.2 == 0.3)            # False (floating point!)
print(math.isclose(0.1 + 0.2, 0.3))  # True

**Your task.**

1. You must ship `n_records` rows in batches of `batch_size`. Compute how many
   batches you need. A partial final batch still counts as one batch. Assign the
   answer to `num_batches` for `n_records = 1003`, `batch_size = 250`.
2. Write `safe_equal(a, b)` that returns `True` when two floats are close enough to
   be considered equal (use `math.isclose`), and test it on `0.1 + 0.2` vs `0.3`.

### Expected output

```text
num_batches: 5
True
False
```


In [ ]:
import math

n_records = 1003
batch_size = 250
# TODO: number of batches, rounding UP for a partial last batch (hint: math.ceil).
num_batches = None
print("num_batches:", num_batches)   # expect 5

def safe_equal(a, b):
    """Return True if a and b are close enough to be treated as equal."""
    # TODO: use math.isclose
    pass

print(safe_equal(0.1 + 0.2, 0.3))     # expect True
print(safe_equal(1.0, 1.5))           # expect False

## Drill 5: `statistics`

**Concept.** The `statistics` module computes summary stats without pandas or numpy:
`mean`, `median`, `mode`, `pstdev` (population standard deviation), and `quantiles`.

Docs: https://docs.python.org/3/library/statistics.html

Why a data engineer cares: you profile data constantly. Median resists outliers
better than mean, and quantiles tell you the shape of a distribution (p25, p50, p75).


In [ ]:
import statistics

amounts = [c["amount"] for c in CLAIMS]
print("mean:  ", round(statistics.mean(amounts), 2))
print("median:", statistics.median(amounts))
print("stdev: ", round(statistics.pstdev(amounts), 2))

**Your task.** Using the claim `amounts` list:

1. Print the mean and median. Note in a comment which one is larger and why that
   hints at a right-skewed distribution (a few large claims pull the mean up).
2. Compute the three quartile cut points with `statistics.quantiles(amounts, n=4)`
   and assign them to `quartiles`. The middle value should equal the median.

### Expected output

```text
mean:   8847.14
median: 7400.0
quartiles (p25, p50, p75): [2600.0, 7400.0, 15800.0]
```


In [ ]:
import statistics

amounts = [c["amount"] for c in CLAIMS]

# TODO: print mean (rounded to 2 dp) and median.
# TODO: which is larger? Add a one-line comment explaining the skew.

quartiles = None   # TODO: statistics.quantiles(amounts, n=4)
print("quartiles (p25, p50, p75):", quartiles)

## Drill 6: `random`

**Concept.** `random` generates pseudo-random values: `random.random()`,
`random.randint(a, b)`, `random.choice(seq)`, `random.sample(seq, k)`, and
`random.shuffle(seq)`. Crucially, `random.seed(n)` makes the sequence reproducible.

Docs: https://docs.python.org/3/library/random.html

Why a data engineer cares: you seed the generator so test data and sampled subsets
are reproducible. An unseeded random test that "sometimes fails" is a nightmare.


In [ ]:
import random

random.seed(42)                      # same seed -> same results every run
print(random.randint(1, 6))          # a dice roll
print(random.choice(["CT", "MA", "NY"]))
print(random.sample(range(100), 3))  # 3 distinct values, no repeats

**Your task.**

1. Seed the generator with `7` so your output is reproducible.
2. Generate a list `sample_ids` of 3 distinct claim ids drawn from `CLAIMS` using
   `random.sample` (sample the claims, then pull their `claim_id`).
3. Generate `fake_amounts`, a list of 5 random claim amounts between 500 and 20000
   (whole dollars), using a loop or a comprehension with `random.randint`.

### Expected output (shape, not exact values)

Unlike the other drills, do not expect specific numbers here. What matters is the
*shape* of the result:

```text
sample_ids: a list of 3 different claim ids from CLAIMS, for example ['CLM-1003', 'CLM-1002', 'CLM-1004']
fake_amounts: a list of 5 whole numbers, each between 500 and 20000
```

The key idea: because you called `random.seed(7)` first, your values will be the
*same every time you run it*, which is exactly what makes a random test
reproducible. Run the cell twice and confirm you get identical output both times.
That is the point of seeding, not the specific digits.


In [ ]:
import random

# TODO: seed with 7 for reproducibility.

all_ids = [c["claim_id"] for c in CLAIMS]
sample_ids = None   # TODO: random.sample(all_ids, 3)
print("sample_ids:", sample_ids)

fake_amounts = []   # TODO: 5 values via random.randint(500, 20000)
print("fake_amounts:", fake_amounts)

## Drill 7: `datetime` and `timedelta`

**Concept.** `datetime` parses, formats, and does arithmetic on dates and times. The
two core moves are `strptime` (string parse time: text into a datetime) and
`strftime` (string format time: datetime into text). `timedelta` represents a
duration you can add or subtract.

Docs: https://docs.python.org/3/library/datetime.html

**Shortcut worth knowing (this answers "why not something simpler?").** If your date
string is already in ISO format (`YYYY-MM-DD`), you can skip the format code entirely:
`date.fromisoformat("2026-05-02")`. Reach for `strptime` only when the format is
something else, for example `05/02/2026` needs `datetime.strptime(s, "%m/%d/%Y")`. We
practice `strptime` in the task below on purpose, because real-world dates are messy
and you will need format codes. But for clean ISO strings, `fromisoformat` is the
simpler tool. Knowing which to use when is the actual skill.

Why a data engineer cares: every event has a timestamp. You parse strings from
files/APIs into real dates, compute durations (how long a claim stayed open), and
format for output. This is the by-hand version of `pd.to_datetime` from Day 4.


In [ ]:
from datetime import datetime, timedelta

opened = datetime.strptime("2026-05-02", "%Y-%m-%d")   # string -> datetime
print(opened.year, opened.strftime("%A"))              # 2026 Saturday
print(opened.strftime("%m/%d/%Y"))                     # 05/02/2026

follow_up = opened + timedelta(days=30)                # add a duration
print("follow-up:", follow_up.date())

**Your task.** For each claim in `CLAIMS`:

1. Parse `opened` and `closed` into `datetime` objects.
2. Compute `days_open` as the integer number of days the claim was open.
3. Build `report`, a list of dicts each with `claim_id`, `days_open`, and
   `opened_weekday` (the weekday name via `strftime("%A")`).
4. After the loop, print the average `days_open` rounded to 1 decimal, and the
   `claim_id` that was open the longest.

### Expected output

```text
{'claim_id': 'CLM-1001', 'days_open': 18, 'opened_weekday': 'Saturday'}
{'claim_id': 'CLM-1002', 'days_open': 22, 'opened_weekday': 'Monday'}
{'claim_id': 'CLM-1003', 'days_open': 3, 'opened_weekday': 'Tuesday'}
{'claim_id': 'CLM-1004', 'days_open': 14, 'opened_weekday': 'Monday'}
{'claim_id': 'CLM-1005', 'days_open': 27, 'opened_weekday': 'Wednesday'}
{'claim_id': 'CLM-1006', 'days_open': 3, 'opened_weekday': 'Tuesday'}
{'claim_id': 'CLM-1007', 'days_open': 4, 'opened_weekday': 'Wednesday'}
average days open: 13.0
longest open: CLM-1005
```


In [ ]:
from datetime import datetime

report = []
for c in CLAIMS:
    # TODO: parse c["opened"] and c["closed"] with datetime.strptime and "%Y-%m-%d".
    # TODO: days_open = (closed - opened).days
    # TODO: append {"claim_id":..., "days_open":..., "opened_weekday":...} to report.
    pass

for row in report:
    print(row)

# TODO: average days_open (rounded to 1 dp) across all rows.
# TODO: claim_id with the maximum days_open (hint: max(report, key=...)).

## Drill 8: `calendar`

**Concept.** `calendar` answers questions about months and weekdays without manual
date math: `calendar.monthrange(year, month)` returns `(first_weekday, num_days)`,
`calendar.isleap(year)` checks leap years, and `calendar.month_name` /
`calendar.day_name` give human labels.

Docs: https://docs.python.org/3/library/calendar.html

Why a data engineer cares: reporting is monthly. "Last day of the month" and "how
many days in this billing period" come up all the time, and February will bite you
if you hardcode 28.


In [ ]:
import calendar

print(calendar.isleap(2024), calendar.isleap(2026))   # True False
first_weekday, num_days = calendar.monthrange(2026, 2)
print("Feb 2026 has", num_days, "days")                # 28
print(calendar.month_name[6], calendar.day_name[0])    # June Monday

**Your task.** Write `last_day_of_month(year, month)` that returns the last calendar
day as an integer (28, 29, 30, or 31). Use `calendar.monthrange`. Then use it to
print the last day for February 2024, February 2026, and April 2026.

### Expected output

```text
Feb 2024: 29
Feb 2026: 28
Apr 2026: 30
```


In [ ]:
import calendar

def last_day_of_month(year, month):
    """Return the last day number of the given month (handles leap years)."""
    # TODO: monthrange returns (first_weekday, num_days). You want num_days.
    pass

print("Feb 2024:", last_day_of_month(2024, 2))   # 29
print("Feb 2026:", last_day_of_month(2026, 2))   # 28
print("Apr 2026:", last_day_of_month(2026, 4))   # 30

## Drill 9: `pathlib`

**Concept.** `pathlib.Path` is the modern, object-oriented way to work with file
paths. You join paths with `/`, inspect parts with `.name`, `.stem`, `.suffix`,
`.parent`, check existence with `.exists()`, and read or write text with
`.read_text()` / `.write_text()`. It works the same on Linux, macOS, and Windows.

Docs: https://docs.python.org/3/library/pathlib.html

Why a data engineer cares: pipelines are full of file paths. `pathlib` avoids fragile
string concatenation and the `os.path.join` dance.


In [ ]:
from pathlib import Path

p = Path("sample_data") / "claims_2026-05.csv"
print("name:  ", p.name)      # claims_2026-05.csv
print("stem:  ", p.stem)      # claims_2026-05
print("suffix:", p.suffix)    # .csv
print("parent:", p.parent)    # sample_data
print("exists:", p.exists())  # True (created in setup)

**Your task.**

1. Build a `Path` to `sample_data` and list only the `.csv` files directly inside it
   (not the archive subfolder) using `Path.glob("*.csv")`. Sort the result and print
   each file's `.stem`.
2. For the May file, extract the month token from the stem. The stem looks like
   `claims_2026-05`; split on `_` and take the last piece. Assign it to `month_token`
   (expected `"2026-05"`).

### Expected output

```text
CSV files:
 - claims_2026-05
 - claims_2026-06
 - claims_2026-07
month_token: 2026-05
```


In [ ]:
from pathlib import Path

data_dir = Path("sample_data")

csv_files = None   # TODO: sorted(data_dir.glob("*.csv"))
print("CSV files:")
# TODO: loop and print each file's .stem

may_file = data_dir / "claims_2026-05.csv"
month_token = None   # TODO: split the .stem on "_" and take the last part
print("month_token:", month_token)   # expect 2026-05

## Drill 10: `glob`

**Concept.** The `glob` module finds paths matching a shell-style wildcard pattern.
`glob.glob("sample_data/*.csv")` matches one level; `glob.glob("sample_data/**/*.csv",
recursive=True)` also descends into subfolders. It returns plain strings (unlike
`Path.glob`, which returns `Path` objects).

Docs: https://docs.python.org/3/library/glob.html

Why a data engineer cares: ingestion often means "grab every file that matches this
pattern in this landing zone". Wildcard matching is the everyday tool for that.


In [ ]:
import glob

# One level deep: the archive subfolder is NOT included.
print("top level:", sorted(glob.glob("sample_data/*.csv")))

# Recursive: ** plus recursive=True descends into archive/.
print("recursive:", sorted(glob.glob("sample_data/**/*.csv", recursive=True)))

**Your task.**

1. Use a recursive glob to collect every `claims_*.csv` file under `sample_data`
   (including the archive) into a sorted list `all_files`. Print how many you found.
2. Read the total number of data rows across all matched files. For each file, open
   it with `csv.DictReader` and add up the row counts into `total_rows`. (Header rows
   are not counted by `DictReader`.)

### Expected output

```text
files found: 4
['sample_data/archive/claims_2026-04.csv', 'sample_data/claims_2026-05.csv', 'sample_data/claims_2026-06.csv', 'sample_data/claims_2026-07.csv']
total_rows: 8
```


In [ ]:
import glob
import csv

all_files = None   # TODO: sorted(glob.glob(... recursive=True)) matching claims_*.csv
print("files found:", len(all_files) if all_files else 0)
print(all_files)

total_rows = 0
# TODO: for each path, open it, read with csv.DictReader, add len(list(reader)) to total_rows.
print("total_rows:", total_rows)   # expect 8 (7 current + 1 archived)

## Drill 11: `itertools`

**Concept.** `itertools` provides fast, memory-friendly building blocks for iteration:

- `chain(a, b)` flattens several iterables into one stream.
- `islice(it, n)` takes the first `n` items (works even on infinite iterators).
- `accumulate(nums)` yields a running total.
- `groupby(sorted_items, key=...)` groups adjacent items that share a key. It only
  groups adjacent runs, so you almost always sort by the same key first.

Docs: https://docs.python.org/3/library/itertools.html

Why a data engineer cares: these are the streaming primitives. Running totals,
batching, and grouping are exactly what you do to large datasets.


In [ ]:
from itertools import accumulate, groupby, islice

nums = [100, 250, 75, 400]
print("running totals:", list(accumulate(nums)))   # [100, 350, 425, 825]
print("first two:", list(islice(nums, 2)))          # [100, 250]

In [ ]:
# groupby needs the data sorted by the same key first.
from itertools import groupby

claims_by_state = sorted(CLAIMS, key=lambda c: c["state"])
for state, group in groupby(claims_by_state, key=lambda c: c["state"]):
    ids = [c["claim_id"] for c in group]
    print(state, ids)

**Your task.**

1. Use `itertools.accumulate` on the claim `amounts` (in list order) to build
   `running_totals`, then print the final cumulative total.
2. Use `sorted` + `itertools.groupby` to build `count_by_line`, a dict mapping each
   insurance `line` ("auto", "home") to the number of claims in it. Remember to sort
   by `line` before grouping.

### Expected output

```text
running_totals: [4200.0, 20000.0, 22600.0, 31700.0, 54000.0, 54530.0, 61930.0]
final total: 61930.0
count_by_line: {'auto': 4, 'home': 3}
```


In [ ]:
from itertools import accumulate, groupby

amounts = [c["amount"] for c in CLAIMS]
running_totals = None   # TODO: list(accumulate(amounts))
print("running_totals:", running_totals)
# print("final total:", running_totals[-1])

count_by_line = {}
# TODO: sort CLAIMS by "line", then groupby "line".
# TODO: for each (line, group), store the number of claims into count_by_line.
print("count_by_line:", count_by_line)

---
# Capstone Challenge: a tiny monthly report pipeline

**Concept.** This pulls together everything above: `glob`/`pathlib` to find files,
`csv` to read them, `datetime` to parse dates, `statistics` and `itertools` to
aggregate, and functions/loops to structure it all. This is a miniature of the
medallion pipeline you build in the main Day 4 lab, done entirely by hand.

**Your task.** Write `monthly_report()` that:

1. Finds every `claims_*.csv` under `sample_data` (recursive).
2. Reads all rows into one combined list of dicts (convert `amount` to `float`).
3. Groups the rows by their `opened` month (the `"YYYY-MM"` prefix) and, for each
   month, computes: `count`, `total_amount`, and `avg_amount` (rounded to 2 dp).
4. Returns a dict keyed by month, sorted so months appear in chronological order.

Expected result (values rounded):

```python
{
  "2026-04": {"count": 1, "total_amount": 3300.0,  "avg_amount": 3300.0},
  "2026-05": {"count": 3, "total_amount": 22600.0, "avg_amount": 7533.33},
  "2026-06": {"count": 3, "total_amount": 31930.0, "avg_amount": 10643.33},
  "2026-07": {"count": 1, "total_amount": 7400.0,  "avg_amount": 7400.0},
}
```


In [ ]:
import glob
import csv
from itertools import groupby
from statistics import mean

def load_all_rows():
    """Read every claims_*.csv under sample_data into one list of dicts."""
    rows = []
    # TODO: recursive glob for claims_*.csv
    # TODO: for each file, DictReader the rows; convert amount to float; append.
    return rows

def monthly_report():
    """Return {month: {count, total_amount, avg_amount}} sorted by month."""
    rows = load_all_rows()
    # TODO: derive a month key "YYYY-MM" from each row's "opened" (first 7 chars).
    # TODO: sort rows by that month key, then groupby it.
    # TODO: for each month build the summary dict; round avg_amount to 2 dp.
    report = {}
    return report

from pprint import pprint
pprint(monthly_report())

---
## Where to go next

- You reused `CLAIMS` and the `sample_data/` files across drills. That is exactly how
  a pipeline is organized: shared inputs, small functions, one step at a time.
- On Day 4's main lab, `pandas` will do the grouping and date parsing you just did by
  hand. You now understand what it is doing under the hood.
- Fast finisher? Rewrite the Capstone Challenge using a `collections.defaultdict`
  instead of `itertools.groupby`, and compare which reads more clearly to you.
